# Customer Credit & Spending — Data Cleaning

**Цель проекта:** выявить некорректные и неполные значения, исправить их, привести к подходящим типам данных.
Проверить:
- пропуски и дубликаты;
- соответсвние значений;
- воставновление значений;


In [1]:
import pandas as pd

df = pd.read_csv("dirty_dataset.csv")

In [2]:
df = pd.read_csv("dirty_dataset.csv")

In [3]:
df.head()

,customer_id,age,income,education,marital_status,credit_score,loan_amount,employment_years,default_status,last_payment_date,phone_number,email,city,state,zip_code,product_type,purchase_frequency,avg_monthly_spend,last_login,account_created
0,CUST001,25.0,45000,Bachelor,Married,720,15000.0,3,0,12/15/2023,-677,john.doe@email.com,New York,NY,10001,credit_card,monthly,850.50,1/15/2024 10:30,3/15/2020
1,CUST002,NaN,75000,Master,Divorced,680,25000.0,8,1,11/20/2023,555-0124,jane.smith@,Chicago,IL,60601,personal_loan,quarterly,1200.75,1/10/2024 14:22,7/22/2019
2,CUST003,28.0,52000,Bachelor,Married,750,18000.0,4,0,12/1/2023,555-0125,bob.wilson@email.com,Los Angeles,CA,90210,credit_card,weekly,650.25,1/20/2024 9:15,1/10/2021
3,CUST004,42.0,95000,PhD,Married,800,35000.0,12,0,12/30/2023,555-0126,alice.brown@email.com,Houston,TX,77001,mortgage,annually,2500.00,1/18/2024 16:45,5/30/2018
4,CUST005,31.0,38000,HighSchool,Single,620,12000.0,2,1,10/15/2023,555-0127,charlie.davis@,Miami,FL,33101,personal_loan,monthly,450.00,1/12/2024 11:30,2/14/2022


In [4]:
# Общий обзор
df.info()
df.describe(include="all")
df.isna().sum()

df.duplicated().sum()
df["customer_id"].duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         50 non-null     str    
 1   age                 49 non-null     float64
 2   income              50 non-null     str    
 3   education           48 non-null     str    
 4   marital_status      48 non-null     str    
 5   credit_score        49 non-null     str    
 6   loan_amount         49 non-null     float64
 7   employment_years    50 non-null     str    
 8   default_status      50 non-null     str    
 9   last_payment_date   49 non-null     str    
 10  phone_number        50 non-null     str    
 11  email               50 non-null     str    
 12  city                50 non-null     str    
 13  state               48 non-null     str    
 14  zip_code            50 non-null     str    
 15  product_type        50 non-null     str    
 16  purchase_frequency  5

np.int64(0)

### Обзор по колонам


In [5]:
# Проверяем распределение возраста и аномальные значения
df["age"].value_counts(dropna=False).sort_index(ascending=False)

# Отрицательные значения возраста переводим в положительные:
df["age"] = df["age"].abs().astype("Int8")

df["age"].isna().sum()

np.int64(1)

In [6]:
# Проверяем значения income и ищем строки с текстом
df["income"].describe()
df["income"].isna().sum()
df["income"].value_counts(dropna=False).sort_index(ascending=False)

income_num = pd.to_numeric(df["income"], errors="coerce")
df.loc[income_num.isna() & df["income"].notna(), "income"]

# Значение '35k' приводим к единому формату
df["income"] = (
    df["income"].astype("string").str.lower().str.replace("k", "000", regex=False)
)

df["income"] = pd.to_numeric(df["income"], errors="coerce").astype("Int32")

In [7]:
# Проверяем education и исправляем различия в написании
df["education"].describe()
df["education"].value_counts(dropna=False)

df["education"] = (
    df["education"].replace("HighSchool", "High School").astype("category")
)

df["education"].value_counts(dropna=False)

education
Bachelor       20
Master         13
Associate       7
High School     4
PhD             4
NaN             2
Name: count, dtype: int64

In [8]:
# Проверяем marital_status и исправляем опечатку
df["marital_status"].describe()
df["marital_status"].value_counts(dropna=False)

df["marital_status"] = (
    df["marital_status"].replace("Divorsed", "Divorced").astype("category")
)

df["marital_status"].value_counts(dropna=False)

marital_status
Married     26
Single      12
Divorced    10
NaN          2
Name: count, dtype: int64

In [9]:
df["credit_score"].describe()
df["credit_score"].isna().sum()
# Проверяем credit_score на текстовые значения
credit_score_num = pd.to_numeric(df["credit_score"], errors="coerce")
df.loc[credit_score_num.isna() & df["credit_score"].notna(), "credit_score"]

# Текстовые значения (например, ERROR) превращаем в пропуски
df["credit_score"] = credit_score_num.astype("Int16")

In [10]:
df["loan_amount"].describe()
df["loan_amount"].isna().sum()
# Проверяем loan_amount на текстовые значения
loan_amount_num = pd.to_numeric(df["loan_amount"], errors="coerce")
df.loc[loan_amount_num.isna() & df["loan_amount"].notna(), "loan_amount"]

df["loan_amount"] = loan_amount_num.astype("Int32")

In [11]:
df["employment_years"].describe()
df["employment_years"].isna().sum()

# Приводим employment_years к числовому типу
employment_years_num = pd.to_numeric(df["employment_years"], errors="coerce")

df.loc[employment_years_num.isna() & df["employment_years"].notna(), "employment_years"]

df["employment_years"] = employment_years_num.astype("Int8")
df["employment_years"].describe()

count        48.0
mean     5.916667
std      3.967063
min           1.0
25%           3.0
50%           5.0
75%           8.0
max          18.0
Name: employment_years, dtype: Float64

In [12]:
df["default_status"].describe()
df["default_status"].isna().sum()

# Приводим default_status к числовому типу и проверяем допустимые значения
default_status_num = pd.to_numeric(df["default_status"], errors="coerce")
df["default_status"] = default_status_num.astype("Int8")

df["default_status"].value_counts(dropna=False)

default_status
0       42
1        7
<NA>     1
Name: count, dtype: Int64

In [13]:
df["last_payment_date"].describe()
df["last_payment_date"].isna().sum()
# Преобразуем дату последнего платежа в datetime.
# Некорректные значения автоматически станут NaT.
df["last_payment_date"] = pd.to_datetime(df["last_payment_date"], errors="coerce")

df["last_payment_date"].isna().sum()

np.int64(1)

In [14]:
# Телефон считаем корректным, если он соответствует формату 555-XXXX
df["phone_number"] = df["phone_number"].astype("string")

valid_phone = df["phone_number"].str.match(r"^555-\d{4}$", na=False)
df.loc[~valid_phone, "phone_number"]

# Исправляем формат вида: (555) 133-0000 -> 555-0133
mask_brackets = df["phone_number"].str.match(r"^\(555\)\s\d{3}-\d{4}$", na=False)

digits = df.loc[mask_brackets, "phone_number"].str.replace(r"\D", "", regex=True)

df.loc[mask_brackets, "phone_number"] = digits.str[:3] + "-0" + digits.str[3:6]

# Исправляем формат вида: 555.0124 -> 555-0124
mask_point = df["phone_number"].str.match(r"^555\.\d{4}$", na=False)

df.loc[mask_point, "phone_number"] = df.loc[mask_point, "phone_number"].str.replace(
    ".", "-", regex=False
)

# Повторно проверяем формат после исправлений.
# Значения, которые невозможно восстановить, заменяем на пропуски.
valid_phone = df["phone_number"].str.match(r"^555-\d{4}$", na=False)
df.loc[~valid_phone, "phone_number"] = pd.NA

In [15]:
# Email считаем корректным, если он соответствует формату name@email.com
df["email"] = df["email"].astype("string")

valid_email = df["email"].str.match(r"^[A-Za-z0-9._]+@email\.com$", na=False)

df.loc[~valid_email, "email"]

# Для адресов, которые заканчиваются на @, восстанавливаем домен email.com
mask_missing_domain = df["email"].str.match(r"^[A-Za-z0-9._]+@$", na=False)

df.loc[mask_missing_domain, "email"] = (
    df.loc[mask_missing_domain, "email"] + "email.com"
)

# Повторная проверка после исправления
valid_email = df["email"].str.match(r"^[A-Za-z0-9._]+@email\.com$", na=False)

# Остальные некорреектные адреса меняем nan
df.loc[~valid_email, "email"] = pd.NA

df["email"].isna().sum()

np.int64(1)

In [16]:
# Проверяем город и штат
df["city"] = df["city"].astype("string").str.strip()
df["state"] = df["state"].astype("string").str.strip()

df["city"].value_counts(dropna=False)
df["state"].value_counts(dropna=False)

print("Уникальных city:", df["city"].nunique(dropna=False))
print("Уникальных state:", df["state"].nunique(dropna=False))

Уникальных city: 46
Уникальных state: 30


In [17]:
# ZIP-код считаем корректным, если он содержит ровно 5 цифр
df["zip_code"] = df["zip_code"].astype("string")

valid_zip = df["zip_code"].str.match(r"^\d{5}$", na=False)
df.loc[~valid_zip, "zip_code"]

# Исправляем формат с двойными кавычками: "12345" -> 12345
mask_zip_quotes = df["zip_code"].str.match(r'^"\d{5}"$', na=False)

df.loc[mask_zip_quotes, "zip_code"] = df.loc[mask_zip_quotes, "zip_code"].str.replace(
    '"', "", regex=False
)

# Повторно проверяем формат после исправления.
# Остальные значения,невозможно восстановить, заменяем на nan.
valid_zip = df["zip_code"].str.match(r"^\d{5}$", na=False)
df.loc[~valid_zip, "zip_code"] = pd.NA

df["zip_code"].isna().sum()

np.int64(1)

In [18]:
# Некорректные категории продукта заменяем на пропуски
df["product_type"].value_counts(dropna=False)

df["product_type"] = (
    df["product_type"].replace(["Unknown", "3"], pd.NA).astype("category")
)

df["product_type"].value_counts(dropna=False)

product_type
personal_loan    20
credit_card      18
mortgage         10
NaN               2
Name: count, dtype: int64

In [19]:
# Исправляем значения частоты покупок
df["purchase_frequency"].value_counts(dropna=False)

df["purchase_frequency"] = (
    df["purchase_frequency"]
    .replace({"30": pd.NA, "bi-weekly": "biweekly"})
    .astype("category")
)

df["purchase_frequency"].value_counts(dropna=False)

purchase_frequency
monthly      22
annually     10
quarterly     8
biweekly      6
weekly        3
NaN           1
Name: count, dtype: int64

In [20]:
# Есть отрицательные значения, меняем значение на положительное и меняем тип данных
df["avg_monthly_spend"].describe()

df["avg_monthly_spend"] = df["avg_monthly_spend"].abs().astype("Float32")

df["avg_monthly_spend"].describe()

count           49.0
mean     1201.775513
std       888.957397
min            380.0
25%            690.5
50%            850.5
75%           1100.0
max           4500.0
Name: avg_monthly_spend, dtype: Float64

In [21]:
# Смена типа данных для даты и время последнего входа
df["last_login"] = pd.to_datetime(df["last_login"], errors="coerce")

df["last_login"].isna().sum()

np.int64(1)

In [22]:
# Смена типа данных для даты создания аккаунта
df["account_created"] = pd.to_datetime(df["account_created"], errors="coerce")

df["account_created"].isna().sum()

np.int64(1)

In [23]:
# Проверка качества данных после очистки
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         50 non-null     str           
 1   age                 49 non-null     Int8          
 2   income              50 non-null     Int32         
 3   education           48 non-null     category      
 4   marital_status      48 non-null     category      
 5   credit_score        48 non-null     Int16         
 6   loan_amount         49 non-null     Int32         
 7   employment_years    48 non-null     Int8          
 8   default_status      49 non-null     Int8          
 9   last_payment_date   49 non-null     datetime64[us]
 10  phone_number        49 non-null     string        
 11  email               49 non-null     string        
 12  city                50 non-null     string        
 13  state               48 non-null     string        
 14  zip_cod

customer_id           0
age                   1
income                0
education             2
marital_status        2
credit_score          2
loan_amount           1
employment_years      2
default_status        1
last_payment_date     1
phone_number          1
email                 1
city                  0
state                 2
zip_code              1
product_type          2
purchase_frequency    1
avg_monthly_spend     1
last_login            1
account_created       1
dtype: int64

In [24]:
# Проверяем итоговые категории и основные диапазоны
df.describe(include="all")

,customer_id,age,income,education,marital_status,credit_score,loan_amount,employment_years,default_status,last_payment_date,phone_number,email,city,state,zip_code,product_type,purchase_frequency,avg_monthly_spend,last_login,account_created
count,50,49.0,50.0,48,48,48.0,49.0,48.0,49.0,49,49,49,50,48,49,48,49,49.0,49,49
unique,50,<NA>,<NA>,5,3,<NA>,<NA>,<NA>,<NA>,NaN,49,49,46,29,49,3,5,<NA>,NaN,NaN
top,CUST001,<NA>,<NA>,Bachelor,Married,<NA>,<NA>,<NA>,<NA>,NaN,555-0124,john.doe@email.com,Miami,CA,10001,personal_loan,monthly,<NA>,NaN,NaN
freq,1,<NA>,<NA>,20,26,<NA>,<NA>,<NA>,<NA>,NaN,1,1,2,6,1,20,22,<NA>,NaN,NaN
mean,NaN,32.816327,62820.0,NaN,NaN,717.5,22877.55102,5.916667,0.142857,2023-12-05 21:33:03.673469,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1201.775513,2024-02-04 10:20:08.571428,2020-06-28 02:56:19.591836
min,NaN,24.0,35000.0,NaN,NaN,600.0,10000.0,1.0,0.0,2023-08-25 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,380.0,2024-01-10 14:22:00,2016-11-14 00:00:00
25%,NaN,28.0,48250.0,NaN,NaN,680.0,16000.0,3.0,0.0,2023-12-05 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,690.5,2024-01-23 10:45:00,2019-05-22 00:00:00
50%,NaN,32.0,58000.0,NaN,NaN,720.0,19000.0,5.0,0.0,2023-12-13 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,850.5,2024-02-04 09:40:00,2020-08-20 00:00:00
75%,NaN,37.0,74750.0,NaN,NaN,750.0,26000.0,8.0,0.0,2023-12-19 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1100.0,2024-02-16 07:15:00,2021-08-09 00:00:00
max,NaN,47.0,125000.0,NaN,NaN,850.0,60000.0,18.0,1.0,2023-12-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4500.0,2024-02-28 15:15:00,2023-05-08 00:00:00



### Итог
Данные очищены и приведены к единому формату. Исправлены некорректные и нестандартизированные значения, обработаны пропуски, приведены типы данных, стандартизированы телефоны, email, ZIP-коды, даты и категориальные признаки. Значения, которые нельзя было надёжно восстановить, заменены на NA.